BIBLIOTECAS

In [ ]:
import pandas as pd
import Levenshtein
from rapidfuzz.distance import DamerauLevenshtein
from thefuzz import fuzz

pd.set_option('display.max_rows', 20)
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import classification_report

LEITURA DOS ARQUIVOS

In [ ]:
caminho_arquivo = 'BDD_etnias_20250505.xlsx';
df_etnias_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'BDD_linguas_20240604.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'descritores.xlsx';
df_etnias_novacod = pd.read_excel(caminho_arquivo, sheet_name='etnias');
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='linguas');

df_ind = pd.read_csv('df_ind.csv', sep=';', encoding='utf-8-sig')
df_ind.info()

QUANTIDADE DE REGISTROS POR ENTRADA NA COLETA

In [ ]:
df_qtd_entradas = df_ind['txt_etnia_entrada_coleta'].value_counts()
df_qtd_entradas

DISTÂNCIA DE LEVENSHTEIN, JARO WINKLER, LEVENSHTEIN-DAMERAU E FUZZ TOKEN_SET_RATIO ENTRE AS ENTRADAS E OS DESCRITORES

In [ ]:
# 1. Pegar apenas os valores únicos e remover valores nulos (NaN) para evitar erros
etnias_entrada = df_ind['txt_etnia_entrada_coleta'].dropna().unique()
etnias_reais = df_etnias_novacod['descricao'].astype(str).str.upper().str.strip().unique()


# 2. Criar uma lista para armazenar os resultados (matriz)
matriz_distancias = []

# 3. Calcular a distância de Levenshtein para cada combinação
for entrada_val in etnias_entrada:
    linha_resultados = []
    
    for real_val in etnias_reais:
        # Calcula a distância entre a string da linha e a string da coluna
        # Convertendo para string (str) por segurança, caso haja números perdidos
        dist = Levenshtein.distance(str(entrada_val), str(real_val))
        
        linha_resultados.append(dist)

        dist = Levenshtein.jaro_winkler(str(entrada_val), str(real_val))
        
        linha_resultados.append(dist)

        dist = DamerauLevenshtein.distance(str(entrada_val), str(real_val))
        
        linha_resultados.append(dist)
        
        dist = fuzz.token_set_ratio(str(entrada_val), str(real_val))
        
        linha_resultados.append(dist)

    matriz_distancias.append(linha_resultados)

# 4. Transformar a matriz no DataFrame final com rótulos
df_distancias_descritores = pd.DataFrame(
    matriz_distancias,
    index=etnias_entrada,     # Rótulos das linhas
    columns=[item for item in etnias_reais for _ in range(4)]   # Rótulos das colunas
)

df_distancias_descritores = df_distancias_descritores.sort_index(axis=0).sort_index(axis=1)
df_distancias_descritores

DISTÂNCIA DE LEVENSHTEIN, JARO WINKLER, DAMERAU LEVENSHTEIN E FUZZ TOKEN_SET_RATIO ENTRE AS ENTRADAS E O TARGET REAL

In [ ]:
# Cria a nova coluna calculando a distância de levenshtein linha por linha 
df_ind['distancia_levenshtein'] = [
    Levenshtein.distance(str(etnia_coleta), str(etnia_nova)) 
    for etnia_coleta, etnia_nova in zip(df_ind['txt_etnia_entrada_coleta'], df_ind['desc_cod_etnia_1_novacod'])
]

# Cria a nova coluna calculando a distância de jaro winkler linha por linha 
df_ind['distancia_jaro_winkler'] = [
    round(Levenshtein.jaro_winkler(str(etnia_coleta), str(etnia_nova)), 5)
    for etnia_coleta, etnia_nova in zip(df_ind['txt_etnia_entrada_coleta'], df_ind['desc_cod_etnia_1_novacod'])
]

# Cria a nova coluna calculando a distância de Damerau-Levenshtein linha por linha 
df_ind['distancia_damerau_levenshtein'] = [
    DamerauLevenshtein.distance(str(etnia_coleta), str(etnia_nova))
    for etnia_coleta, etnia_nova in zip(df_ind['txt_etnia_entrada_coleta'], df_ind['desc_cod_etnia_1_novacod'])
]

# Cria a nova coluna calculando a distância de fuzz.token_set_ratio linha por linha 
df_ind['token_set_ratio'] = [
    fuzz.token_set_ratio(str(etnia_coleta), str(etnia_nova))
    for etnia_coleta, etnia_nova in zip(df_ind['txt_etnia_entrada_coleta'], df_ind['desc_cod_etnia_1_novacod'])
]

df_ind

MODELO SIMPLES LEVENSHTEIN PARA IDENTIFICAR A STRING DESCRITORA MAIS PRÓXIMA DA STRING DE ENTRADA

In [ ]:
# 1. Calcular a distância de Levenshtein entre cada entrada e todos os descritores
df_ind['levenshtein_min'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: min(
        [Levenshtein.distance(str(entrada), str(descritor)) for descritor in etnias_reais]
    ) if pd.notna(entrada) else None
)

# 2. Encontrar o descritor mais próximo para cada entrada
df_ind['descricao_mais_proxima_levenshtein'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: min(
        etnias_reais,
        key=lambda descritor: Levenshtein.distance(str(entrada), str(descritor))
    ) if pd.notna(entrada) else None
)

df_ind

MÉTRICAS MODELO SIMPLES LEVENSHTEIN

In [ ]:
y_treino = df_ind['desc_cod_etnia_1_novacod']
y_pred_treino = df_ind['descricao_mais_proxima_levenshtein']


print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

MODELO SIMPLES JARO WINKLER PARA IDENTIFICAR A STRING DESCRITORA MAIS PRÓXIMA DA STRING DE ENTRADA

In [ ]:
# 1. Calcular a distância de Jaro-Winkler entre cada entrada e todos os descritores
df_ind['jaro_winkler_max'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: max(
        [Levenshtein.jaro_winkler(str(entrada), str(descritor)) for descritor in etnias_reais]
    ) if pd.notna(entrada) else None
)

# 2. Encontrar o descritor mais próximo para cada entrada
df_ind['descricao_mais_proxima_jaro_winkler'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: max(
        etnias_reais,
        key=lambda descritor: Levenshtein.jaro_winkler(str(entrada), str(descritor))
    ) if pd.notna(entrada) else None
)

df_ind

MÉTRICAS MODELO SIMPLES JARO WINKLER

In [ ]:
y_treino = df_ind['desc_cod_etnia_1_novacod']
y_pred_treino = df_ind['descricao_mais_proxima_jaro_winkler']


print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

MODELO SIMPLES DAMERAU-LEVENSHTEIN PARA IDENTIFICAR A STRING DESCRITORA MAIS PRÓXIMA DA STRING DE ENTRADA

In [ ]:
# 1. Calcular a distância de Levenshtein entre cada entrada e todos os descritores
df_ind['levenshtein_damerau_min'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: min(
        [DamerauLevenshtein.distance(str(entrada), str(descritor)) for descritor in etnias_reais]
    ) if pd.notna(entrada) else None
)

# 2. Encontrar o descritor mais próximo para cada entrada
df_ind['descricao_mais_proxima_levenshtein_damerau'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: min(
        etnias_reais,
        key=lambda descritor: DamerauLevenshtein.distance(str(entrada), str(descritor))
    ) if pd.notna(entrada) else None
)

df_ind

MÉTRICAS MODELO SIMPLES DAMERAU-LEVENSHTEIN

In [ ]:
y_treino = df_ind['desc_cod_etnia_1_novacod']
y_pred_treino = df_ind['descricao_mais_proxima_levenshtein_damerau']


print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

MODELO SIMPLES FUZZ SET_TOKEN_RATIO PARA IDENTIFICAR A STRING DESCRITORA MAIS PRÓXIMA DA STRING DE ENTRADA

In [ ]:
# 1. Calcular a distância de Levenshtein entre cada entrada e todos os descritores
df_ind['fuzz_set_token_ratio_max'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: max(
        [fuzz.token_set_ratio(str(entrada), str(descritor)) for descritor in etnias_reais]
    ) if pd.notna(entrada) else None
)

# 2. Encontrar o descritor mais próximo para cada entrada
df_ind['descricao_mais_proxima_fuzz_set_token_ratio'] = df_ind['txt_etnia_entrada_coleta'].apply(
    lambda entrada: max(
        etnias_reais,
        key=lambda descritor: fuzz.token_set_ratio(str(entrada), str(descritor))
    ) if pd.notna(entrada) else None
)

df_ind

MÉTRICAS MODELO SIMPLES FUZZ SET_TOKEN_RATIO

In [ ]:
y_treino = df_ind['desc_cod_etnia_1_novacod']
y_pred_treino = df_ind['descricao_mais_proxima_fuzz_set_token_ratio']


print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

ARMAZENAMENTO DOS DADOS EM UM ARQUIVO CSV

In [ ]:
df_ind.to_csv('df_ind_distancia_descritores.csv', index=False, sep=';', encoding='utf-8-sig')

ARMAZENAMENTO DOS DADOS COM A MELHOR MÉTRICA DE DISTÂNCIA EM UM ARQUIVO CSV

In [ ]:
df_ind = df_ind.drop(columns=['levenshtein_min','descricao_mais_proxima_levenshtein'])
df_ind = df_ind.drop(columns=['jaro_winkler_max','descricao_mais_proxima_jaro_winkler'])
# df_ind = df_ind.drop(columns=['levenshtein_damerau_min','descricao_mais_proxima_levenshtein_damerau'])
df_ind = df_ind.drop(columns=['fuzz_set_token_ratio_max','descricao_mais_proxima_fuzz_set_token_ratio'])
df_ind = df_ind.drop(columns=['distancia_levenshtein','distancia_jaro_winkler','distancia_damerau_levenshtein','token_set_ratio'])

df_ind = df_ind.rename(columns={'levenshtein_damerau_min':'distancia'})
df_ind = df_ind.rename(columns={'descricao_mais_proxima_levenshtein_damerau':'descricao_mais_proxima'})

df_ind.to_csv('df_ind_distancia.csv', index=False, sep=';', encoding='utf-8-sig')